In [0]:
-- ============================================================================
-- GLOBAL CLOUD INFRASTRUCTURE SETUP
-- ============================================================================

-- 1. Create a secure storage link to your AWS S3 bucket
CREATE OR REPLACE STORAGE INTEGRATION s3_indian_retail_integration
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = 'S3'
  ENABLED = TRUE
  STORAGE_ALLOWED_LOCATIONS = ('s3://indian-retail-analytics-landing-zone/landing/')
  STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::767397728224:role/Snowflake_Storage_Integration_Role';

-- Verify properties to complete the AWS IAM Role trust handshake
DESCRIBE INTEGRATION s3_indian_retail_integration;


-- ============================================================================
-- DATABASE & MEDALLION LAYER ARCHITECTURE
-- ============================================================================

CREATE OR REPLACE DATABASE indian_retail_db;

CREATE OR REPLACE SCHEMA indian_retail_db.bronze;
CREATE OR REPLACE SCHEMA indian_retail_db.silver;
CREATE OR REPLACE SCHEMA indian_retail_db.gold;


-- ============================================================================
-- BRONZE LAYER (RAW DATA INGESTION & DATA LAKE LINK)
-- ============================================================================

-- 1. Create the External Stage pointing to the S3 Landing Zone
CREATE OR REPLACE STAGE indian_retail_db.bronze.s3_landing_stage
  STORAGE_INTEGRATION = s3_indian_retail_integration
  URL = 's3://indian-retail-analytics-landing-zone/landing/';

-- 2. Create Split Semi-Structured Bronze Tables with 30-Day Time Travel Enabled
CREATE OR REPLACE TABLE indian_retail_db.bronze.stg_raw_sales (
    raw_data VARIANT,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
) DATA_RETENTION_TIME_IN_DAYS = 30;

CREATE OR REPLACE TABLE indian_retail_db.bronze.stg_raw_customers (
    raw_data VARIANT,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
) DATA_RETENTION_TIME_IN_DAYS = 30;

CREATE OR REPLACE TABLE indian_retail_db.bronze.stg_raw_products (
    raw_data VARIANT,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
) DATA_RETENTION_TIME_IN_DAYS = 30;


-- ============================================================================
-- CHANGE DATA CAPTURE (CDC STREAMS FOR DATABRICKS HANDSHAKE)
-- ============================================================================

CREATE OR REPLACE STREAM indian_retail_db.bronze.sales_cdc_stream ON TABLE indian_retail_db.bronze.stg_raw_sales;
CREATE OR REPLACE STREAM indian_retail_db.bronze.customers_cdc_stream ON TABLE indian_retail_db.bronze.stg_raw_customers;
CREATE OR REPLACE STREAM indian_retail_db.bronze.products_cdc_stream ON TABLE indian_retail_db.bronze.stg_raw_products;


-- ============================================================================
-- AUTOMATED INGESTION PIPES (SNOWPIPE RUNTIME ENGINES)
-- ============================================================================

-- 1. Automated Continuous SALES Ingestion Engine
CREATE OR REPLACE PIPE indian_retail_db.bronze.pipe_sales
AUTO_INGEST = TRUE
AS
COPY INTO indian_retail_db.bronze.stg_raw_sales (raw_data) FROM (
  SELECT OBJECT_CONSTRUCT(
    'Order_ID', $1, 'Customer_ID', $2, 'Product_ID', $3, 'Order_Date', $4, 'Order_Time', $5,
    'Delivery_Date', $6, 'Quantity', $7, 'Unit_Price', $8, 'Order_Value', $9, 'Shipping_Cost', $10,
    'Coupon_Code', $11, 'Coupon_Discount', $12, 'Total_Amount', $13, 'Payment_Mode', $14,
    'Order_Status', $15, 'Rating', $16, 'Review_Text', $17, 'City', $18, 'State', $19,
    'Customer_Age', $20, 'Customer_Age_Group', $21
  ) FROM @indian_retail_db.bronze.s3_landing_stage
)
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1)
PATTERN = '.*sales.*\.csv';

-- 2. Automated Continuous PRODUCTS Ingestion Engine
CREATE OR REPLACE PIPE indian_retail_db.bronze.pipe_products
AUTO_INGEST = TRUE
AS
COPY INTO indian_retail_db.bronze.stg_raw_products (raw_data) FROM (
  SELECT OBJECT_CONSTRUCT(
    'Product_ID', $1, 'Product_Name', $2, 'Category', $3, 'Brand', $4, 'Original_Price', $5,
    'Discount_Percent', $6, 'Discount_Amount', $7, 'Selling_Price', $8, 'Stock_Quantity', $9,
    'Weight_kg', $10, 'Avg_Rating', $11, 'Total_Reviews', $12
  ) FROM @indian_retail_db.bronze.s3_landing_stage
)
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1)
PATTERN = '.*products.*\.csv';

-- 3. Automated Continuous CUSTOMERS Ingestion Engine
CREATE OR REPLACE PIPE indian_retail_db.bronze.pipe_customers
AUTO_INGEST = TRUE
AS
COPY INTO indian_retail_db.bronze.stg_raw_customers (raw_data) FROM (
  SELECT OBJECT_CONSTRUCT(
    'Customer_ID', $1, 'Customer_Name', $2, 'Gender', $3, 'Age', $4, 'Age_Group', $5,
    'Date_of_Birth', $6, 'Email', $7, 'Phone', $8, 'City', $9, 'State', $10, 'Pincode', $11,
    'Registration_Date', $12, 'Customer_Tier', $13, 'Total_Orders', $14, 'Total_Spent', $15
  ) FROM @indian_retail_db.bronze.s3_landing_stage
)
FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1)
PATTERN = '.*customers.*\.csv';


-- ============================================================================
-- GOLD LAYER ARCHITECTURE (ANALYTICS-READY STAR SCHEMA)
-- ============================================================================

CREATE OR REPLACE TABLE indian_retail_db.gold.fact_sales (
    order_id STRING,
    customer_id STRING,
    product_id STRING,
    order_date DATE,
    quantity INT,
    unit_price NUMBER(10,2),
    total_amount NUMBER(12,2),
    payment_mode STRING,
    order_status STRING
) DATA_RETENTION_TIME_IN_DAYS = 30; 

CREATE OR REPLACE TABLE indian_retail_db.gold.dim_customers (
    customer_id STRING,
    customer_name STRING,
    gender STRING,
    customer_tier STRING,
    city STRING,
    state STRING
) DATA_RETENTION_TIME_IN_DAYS = 30;

CREATE OR REPLACE TABLE indian_retail_db.gold.dim_products (
    product_id STRING,
    product_name STRING,
    category STRING,
    brand STRING
) DATA_RETENTION_TIME_IN_DAYS = 30;


-- ============================================================================
-- PIPELINE AUTOMATION & INCREMENTAL SCHEDULER
-- ============================================================================

CREATE OR REPLACE TASK indian_retail_db.gold.sync_gold_star_schema_task
WAREHOUSE = COMPUTE_WH
SCHEDULE = '30 MINUTE'
AS
EXECUTE IMMEDIATE '
BEGIN
    -- A. Upsert records into FACT_SALES using case-preserved Python identifiers
    MERGE INTO indian_retail_db.gold.fact_sales target
    USING indian_retail_db.silver.one_big_table source
    ON target.order_id = source."order_id"
    WHEN NOT MATCHED THEN
    INSERT (order_id, customer_id, product_id, order_date, quantity, unit_price, total_amount, payment_mode, order_status)
    VALUES (source."order_id", source."customer_id", source."product_id", source."order_date", source."quantity", source."unit_price", source."total_amount", source."payment_mode", source."order_status");

    -- B. Upsert records into DIM_CUSTOMERS
    MERGE INTO indian_retail_db.gold.dim_customers target
    USING (SELECT DISTINCT "customer_id", "customer_name", "gender", "customer_tier", "city", "state" FROM indian_retail_db.silver.one_big_table) source
    ON target.customer_id = source."customer_id"
    WHEN NOT MATCHED THEN
    INSERT (customer_id, customer_name, gender, customer_tier, city, state)
    VALUES (source."customer_id", source."customer_name", source."gender", source."customer_tier", source."city", source."state");

    -- C. Upsert records into DIM_PRODUCTS
    MERGE INTO indian_retail_db.gold.dim_products target
    USING (SELECT DISTINCT "product_id", "product_name", "category", "brand" FROM indian_retail_db.silver.one_big_table) source
    ON target.product_id = source."product_id"
    WHEN NOT MATCHED THEN
    INSERT (product_id, product_name, category, brand)
    VALUES (source."product_id", source."product_name", source."category", source."brand");
END;
';

-- Turn the scheduler on
ALTER TASK indian_retail_db.gold.sync_gold_star_schema_task RESUME;


-- ============================================================================
-- AUDITING AND QUALITY ASSURANCE QUERIES (RUN AS NEEDED)
-- ============================================================================

-- Check live row queue counts
-- SELECT COUNT(*) FROM indian_retail_db.bronze.sales_cdc_stream;

-- Check final Gold layer storage metrics
-- SELECT COUNT(*) FROM indian_retail_db.gold.fact_sales;
